In [1]:
import torch
import torch.nn as nn
import numpy as np
from torch import Tensor
from matplotlib import pyplot as plt
from sklearn.datasets import make_moons, make_circles

In [2]:
class Flow(nn.Module): 
    '''
    Defines the architecture for flow matching
    '''

    def __init__(self, dim: int = 6, h: int = 192): # fully connected neural network for learning velocity field 
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(dim + 1, h),  
            nn.ELU(),              
            nn.Linear(h, h),       
            nn.ELU(),             
            nn.Linear(h, h),      
            nn.ELU(),              
            nn.Linear(h, dim)       
        )
    
    def forward(self, t: Tensor, xt: Tensor) -> Tensor:
        '''
        Concatenates time and position vectors for forward pass through neural network  
        '''

        combined_input = torch.cat((t, xt), -1)

        return self.net(combined_input)
    
    def step(self, xt: Tensor, t_start: Tensor, t_end: Tensor) -> Tensor: 
        '''
        Performs midpoint integration to sample transform x0 to x1 (i.e., sample from p1, starting from p0)
        '''
      
        t_start = t_start.view(1, 1).expand(xt.shape[0], 1)
        midpoint_time = t_start + (t_end - t_start) / 2 # evalute the midpoint 
        intermediate_value = self(xt=xt, t=t_start) # compute the velocity at this point using the trained neural network 
        
        updated_xt = xt + intermediate_value * (t_end - t_start) / 2 # move xt using computed velocity 
        midpoint_output = self(t=midpoint_time, xt=updated_xt) # compute vt again 
        
        return xt + (t_end - t_start) * midpoint_output # compute xt

In [3]:
# Training loop
def train():
    '''
    Defines the training procedure for the neural network
    '''
    for _ in range(1000000): #10000000
        if _ == 0:  
            x1 = Tensor([1, 2, 3, 4, 5, 6]).unsqueeze(0)
     
        x0 = torch.randn_like(x1)
        t = torch.rand(len(x1), 1) #torch.rand(1).unsqueeze(0) #

        xt = (1 - t) * x0 + t * x1

        # Compute loss 
        loss = loss_fn(flow(t=t, xt=xt), x1 - x0)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        print(f"Loss at epoch {_}: {loss.item()}")

        if loss.item() < 0.0001:
            break 

######################################################
# Model training 
######################################################
# Initialize model and optimizer
flow = Flow()
optimizer = torch.optim.Adam(flow.parameters(), 1e-3)
loss_fn = nn.MSELoss()
train()


Loss at epoch 0: 12.625760078430176
Loss at epoch 1: 17.01517677307129
Loss at epoch 2: 17.901391983032227
Loss at epoch 3: 9.33548641204834
Loss at epoch 4: 11.407063484191895
Loss at epoch 5: 5.718137264251709
Loss at epoch 6: 6.540640354156494
Loss at epoch 7: 9.693666458129883
Loss at epoch 8: 1.327805757522583
Loss at epoch 9: 1.6271562576293945
Loss at epoch 10: 2.996583938598633
Loss at epoch 11: 2.8454036712646484
Loss at epoch 12: 5.76554536819458
Loss at epoch 13: 1.3223381042480469
Loss at epoch 14: 4.38132905960083
Loss at epoch 15: 1.604899525642395
Loss at epoch 16: 8.178162574768066
Loss at epoch 17: 6.916646957397461
Loss at epoch 18: 20.285642623901367
Loss at epoch 19: 2.6640889644622803
Loss at epoch 20: 4.830813884735107
Loss at epoch 21: 0.32707345485687256
Loss at epoch 22: 7.652897357940674
Loss at epoch 23: 0.5731697678565979
Loss at epoch 24: 23.344497680664062
Loss at epoch 25: 1.539009690284729
Loss at epoch 26: 2.744438409805298
Loss at epoch 27: 2.583589553

KeyboardInterrupt: 

In [ ]:

"""

######################################################
# Model sampling
######################################################
print("Begin Inference")
x = torch.randn(3000, 2)
n_steps = 8
fig, axes = plt.subplots(1, n_steps + 1, figsize=(30, 4), sharex=True, sharey=True)
time_steps = torch.linspace(0, 1.0, n_steps + 1)


axes[0].scatter(x.detach()[:, 0], x.detach()[:, 1], s=10, alpha=0.50)
axes[0].set_title(f't = {time_steps[0]:.2f}')
axes[0].set_xlim(-2.0, 2.0)
axes[0].set_ylim(-2.0, 2.0)

for i in range(n_steps):
    x = flow.step(xt=x, t_start=time_steps[i], t_end=time_steps[i + 1]) # sample xt from neural network 
    axes[i + 1].scatter(x.detach()[:, 0], x.detach()[:, 1], s=10, alpha=0.50)
    axes[i + 1].set_title(f't = {time_steps[i + 1]:.2f}')

# save as .svg
plt.savefig('q_p.svg', format='svg', dpi=300, bbox_inches='tight')

######################################################
# Generate p0 (checker board)
######################################################
# Generate data for a dummy distribution (testing purposes only)
rv1 = torch.rand(10000) * 4 - 2 
rv2 = torch.rand(10000) * 4 - 2 
mask = (torch.floor(rv1) + torch.floor(rv2)) % 2 == 0  

x1 = torch.stack((rv1[mask], rv2[mask]), dim=1)
x1_np = x1.numpy()

# Plot the checkerboard distribution
plt.figure(figsize=(4, 4))
plt.scatter(x1_np[:, 0], x1_np[:, 1], s=10, alpha=0.50)
plt.title("Checkerboard Distribution")
plt.xlabel("X1")
plt.ylabel("X2")
plt.xlim(-2, 2)
plt.ylim(-2, 2)
# save as .svg
plt.savefig('p_grnd.svg', format='svg', dpi=300, bbox_inches='tight')

plt.show()

"""


'\n\n######################################################\n# Model sampling\n######################################################\nprint("Begin Inference")\nx = torch.randn(3000, 2)\nn_steps = 8\nfig, axes = plt.subplots(1, n_steps + 1, figsize=(30, 4), sharex=True, sharey=True)\ntime_steps = torch.linspace(0, 1.0, n_steps + 1)\n\n\naxes[0].scatter(x.detach()[:, 0], x.detach()[:, 1], s=10, alpha=0.50)\naxes[0].set_title(f\'t = {time_steps[0]:.2f}\')\naxes[0].set_xlim(-2.0, 2.0)\naxes[0].set_ylim(-2.0, 2.0)\n\nfor i in range(n_steps):\n    x = flow.step(xt=x, t_start=time_steps[i], t_end=time_steps[i + 1]) # sample xt from neural network \n    axes[i + 1].scatter(x.detach()[:, 0], x.detach()[:, 1], s=10, alpha=0.50)\n    axes[i + 1].set_title(f\'t = {time_steps[i + 1]:.2f}\')\n\n# save as .svg\nplt.savefig(\'q_p.svg\', format=\'svg\', dpi=300, bbox_inches=\'tight\')\n\n######################################################\n# Generate p0 (checker board)\n############################